# 🔄 Day 03: Linked List Reversal Patterns — Beyond the Basics

---

## You Know How to Reverse a Full List. Now What?

Reversing the entire list is just the warmup. The real interview questions ask you to reverse **parts** of the list:

- Reverse nodes from position `left` to `right` (LC #92)
- Swap every pair of nodes (LC #24)
- Reverse in groups of k (LC #25 — the hard one)

All of these build on the same prev/curr/next reversal mechanics from Day 01. The tricky part is managing the **connections** — the nodes before and after the reversed section need to be reconnected properly.

Let me walk you through this step by step.

---

In [ ]:
# Helpers
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build_list(values):
    dummy = ListNode(0)
    curr = dummy
    for v in values:
        curr.next = ListNode(v)
        curr = curr.next
    return dummy.next

def print_list(head):
    parts = []
    while head:
        parts.append(str(head.val))
        head = head.next
    print(" → ".join(parts) + " → None")

---

## 🎯 Problem 1: Reverse Linked List II (LC #92)

Reverse only the nodes from position `left` to `right` (1-indexed).

```
Input:  1 → 2 → 3 → 4 → 5,  left=2, right=4
Output: 1 → 4 → 3 → 2 → 5
             ↑ reversed ↑
```

**The plan:**
1. Walk to the node just BEFORE `left` — call it `before_left`
2. Reverse the sublist from `left` to `right`
3. Reconnect: `before_left → reversed_head` and `reversed_tail → after_right`

```
Before reversal:
  [before_left] → [left] → ... → [right] → [after_right]

After reversal:
  [before_left] → [right] → ... → [left] → [after_right]
```

In [ ]:
def reverseBetween(head, left, right):
    """
    Reverse nodes between positions left and right.
    Time: O(n), Space: O(1)
    """
    if left == right:
        return head  # Nothing to reverse
    
    dummy = ListNode(0)
    dummy.next = head
    
    # Step 1: Walk to the node just before 'left'
    before_left = dummy
    for _ in range(left - 1):
        before_left = before_left.next
    
    # Step 2: Reverse (right - left) links
    # 'prev' starts at node at position 'left'
    prev = before_left.next  # This will become the TAIL of reversed section
    curr = prev.next
    
    for _ in range(right - left):
        # Pull curr out and insert it right after before_left
        next_temp = curr.next
        curr.next = before_left.next
        before_left.next = curr
        prev.next = next_temp
        curr = next_temp
    
    return dummy.next

lst = build_list([1, 2, 3, 4, 5])
print("Before:", end=" "); print_list(build_list([1,2,3,4,5]))
result = reverseBetween(lst, 2, 4)
print("After: ", end=" "); print_list(result)  # 1→4→3→2→5

The trick with partial reversal is the "insert at front" technique — instead of flipping arrows in place, you repeatedly pull the next node and insert it at the front of the reversed section. It's cleaner and less error-prone.

---

## 🎯 Problem 2: Swap Nodes in Pairs (LC #24)

Swap every two adjacent nodes.

```
Input:  1 → 2 → 3 → 4 → None
Output: 2 → 1 → 4 → 3 → None
```

**Visual:**
```
Before: prev → [a] → [b] → rest
After:  prev → [b] → [a] → rest
Then:   prev moves to [a], repeat
```

In [ ]:
def swapPairs(head):
    """
    Swap nodes in pairs.
    Time: O(n), Space: O(1)
    """
    dummy = ListNode(0)
    dummy.next = head
    prev = dummy
    
    while prev.next and prev.next.next:
        a = prev.next     # First node of pair
        b = a.next        # Second node of pair
        
        # Swap: prev → b → a → rest
        prev.next = b
        a.next = b.next
        b.next = a
        
        prev = a  # Move to next pair
    
    return dummy.next

lst = build_list([1, 2, 3, 4])
print("Before:", end=" "); print_list(build_list([1,2,3,4]))
result = swapPairs(lst)
print("After: ", end=" "); print_list(result)  # 2→1→4→3

# Odd count — last node stays
lst2 = build_list([1, 2, 3])
result2 = swapPairs(lst2)
print("Odd:   ", end=" "); print_list(result2)  # 2→1→3

---

## 🎯 Problem 3: Reverse Nodes in k-Group (LC #25) — Hard

This is the boss fight of linked list reversal problems. Reverse every group of k nodes. If remaining nodes are fewer than k, leave them as-is.

```
Input:  1 → 2 → 3 → 4 → 5,  k = 3
Output: 3 → 2 → 1 → 4 → 5
        (first 3 reversed, last 2 left alone)
```

**Strategy:**
1. Check if there are k nodes remaining (walk ahead k steps)
2. If yes, reverse those k nodes
3. Connect the reversed group to the previous section
4. Move to the next group, repeat

This is where the dummy head, group reversal, and pointer reconnection all come together.

In [ ]:
def reverseKGroup(head, k):
    """
    Reverse nodes in groups of k.
    Time: O(n), Space: O(1)
    """
    dummy = ListNode(0)
    dummy.next = head
    group_prev = dummy  # Node before current group
    
    while True:
        # Check if k nodes remain
        kth = group_prev
        for _ in range(k):
            kth = kth.next
            if not kth:
                return dummy.next  # Less than k remaining, we're done
        
        group_next = kth.next  # Node after current group
        
        # Reverse k nodes: group_prev → [a→b→c] → group_next
        prev = group_next  # Reversed section should point to next group
        curr = group_prev.next
        
        for _ in range(k):
            next_temp = curr.next
            curr.next = prev
            prev = curr
            curr = next_temp
        
        # Reconnect: group_prev → reversed section
        first_of_group = group_prev.next  # Was first, now last after reversal
        group_prev.next = prev            # prev is now head of reversed group (was kth)
        group_prev = first_of_group       # Move to end of reversed group for next iteration
    
    return dummy.next

lst = build_list([1, 2, 3, 4, 5])
print("k=3:", end=" "); print_list(reverseKGroup(lst, 3))  # 3→2→1→4→5

lst2 = build_list([1, 2, 3, 4, 5, 6])
print("k=2:", end=" "); print_list(reverseKGroup(lst2, 2))  # 2→1→4→3→6→5

lst3 = build_list([1, 2, 3, 4, 5])
print("k=1:", end=" "); print_list(reverseKGroup(lst3, 1))  # 1→2→3→4→5 (no change)

---

## 🗺️ Reversal Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  REVERSAL PATTERNS                                                   ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. FULL REVERSE: prev/curr/next — basic building block             ║
║                                                                      ║
║  2. PARTIAL REVERSE (left to right): walk to before_left, then      ║
║     repeatedly pull & insert at front of reversed section            ║
║                                                                      ║
║  3. SWAP PAIRS: prev → b → a → rest, advance prev by 2             ║
║     (Special case of k-group with k=2)                               ║
║                                                                      ║
║  4. K-GROUP: Check k remaining → reverse group → reconnect          ║
║     Track group_prev (before group) and group_next (after group)    ║
║                                                                      ║
║  Common thread: DUMMY HEAD + careful reconnection after reversal    ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Pattern |
|---|---------|------------|--------|
| 206 | Reverse Linked List | Easy | Full reverse |
| 92 | Reverse Linked List II | Medium | Partial reverse |
| 24 | Swap Nodes in Pairs | Medium | Pair swap |
| 25 | Reverse Nodes in k-Group | Hard | K-group reverse |
| 143 | Reorder List | Medium | Middle + reverse + merge |
| 61 | Rotate List | Medium | Find tail + partial reverse |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Partial reversal: walk to the boundary, then reverse inside    │
├──────────────────────────────────────────────────────────────────────┤
│  📌 "Insert at front" technique is cleaner than in-place flipping  │
├──────────────────────────────────────────────────────────────────────┤
│  📌 K-group: always check if k nodes remain before reversing       │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Draw it. Every. Single. Time. Pointer bugs are visual bugs.    │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Dummy head + group_prev/group_next makes reconnection safe     │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Merge & Sort Patterns

Merging sorted lists and sorting a linked list with merge sort.

---

### 🎉 Reversal Patterns Complete!
Full, partial, pairs, k-groups — you can reverse anything now.